# 07 — Bayesian Model Checking & Comparison
**References:** Gelman et al. BDA3 Ch. 6–7 · Vehtari, Gelman & Gabry (2017) · Kass & Raftery (1995)

## Narrative thread
```
Posterior predictive checks -> Bayesian p-values -> Bayes factors -> WAIC -> LOO-CV -> Model selection
```

## 1. Posterior predictive checks (PPCs)

**Idea:** If the model is correct, replicated data $\mathbf{x}^{\text{rep}}$ should look like the observed data $\mathbf{x}$.

**Algorithm:**
1. Draw $\theta^{(s)} \sim p(\theta \mid \mathbf{x})$ from the posterior
2. Draw $\mathbf{x}^{\text{rep},(s)} \sim p(\mathbf{x}^{\text{rep}} \mid \theta^{(s)})$
3. Compute a **test statistic** $T(\mathbf{x}^{\text{rep},(s)})$ for each replicate
4. Compare the distribution of $T(\mathbf{x}^{\text{rep}})$ to the observed $T(\mathbf{x})$

**Bayesian p-value:**
$$p_B = P(T(\mathbf{x}^{\text{rep}}) \geq T(\mathbf{x}) \mid \mathbf{x})$$

Values near 0 or 1 signal model misfit. Values near 0.5 indicate good fit for that test statistic.

> **Key difference from frequentist:** PPCs use the posterior-averaged predictive distribution — parameter uncertainty is accounted for. The check is specific to a chosen test statistic; different statistics test different aspects of fit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

# ── PPC: Normal model fitted to skewed data ────────────────────────────────
# True DGP: log-normal. Model: normal. PPC should detect misfit.
np.random.seed(42)
n_data    = 80
y_obs     = np.random.lognormal(mean=1.0, sigma=0.5, size=n_data)  # skewed data

# Fit Normal model (MLE via posterior with flat prior)
mu_hat    = y_obs.mean()
sigma_hat = y_obs.std(ddof=1)

# Posterior for mu (known sigma = sigma_hat, flat prior on mu)
# mu | y ~ N(mu_hat, sigma_hat^2/n)
post_mu_mean = mu_hat
post_mu_sd   = sigma_hat / np.sqrt(n_data)

# Generate posterior predictive replicates
S = 2000
mu_draws    = np.random.normal(post_mu_mean, post_mu_sd, S)
y_rep       = np.random.normal(mu_draws[:, None], sigma_hat, (S, n_data))

# Test statistics
T_skew_obs  = stats.skew(y_obs)
T_max_obs   = y_obs.max()
T_skew_rep  = stats.skew(y_rep, axis=1)
T_max_rep   = y_rep.max(axis=1)

pB_skew = (T_skew_rep >= T_skew_obs).mean()
pB_max  = (T_max_rep  >= T_max_obs).mean()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Density of observed vs replicates
x_range = np.linspace(y_obs.min()-1, y_obs.max()+1, 300)
ax0 = axes[0]
for i in range(0, S, 100):  # plot 20 replicates
    ax0.plot(x_range, stats.gaussian_kde(y_rep[i])(x_range), color='#4361ee', lw=0.5, alpha=0.2)
ax0.plot(x_range, stats.gaussian_kde(y_obs)(x_range), color='#f72585', lw=2.5, label='Observed')
ax0.set_title('PPC: density overlay\nNormal model misses right tail')
ax0.set_xlabel('y'); ax0.legend()

# Skewness distribution
ax1 = axes[1]
ax1.hist(T_skew_rep, bins=50, density=True, color='#4361ee', alpha=0.7, edgecolor='white')
ax1.axvline(T_skew_obs, color='#f72585', lw=2.5, linestyle='--', label=f'Observed skew={T_skew_obs:.2f}')
ax1.set_title(f'PPC: skewness\nBayesian p-value = {pB_skew:.3f}\n(near 0 → model too symmetric)')
ax1.set_xlabel('Skewness of replicate'); ax1.legend(fontsize=8)

# Max value distribution
ax2 = axes[2]
ax2.hist(T_max_rep, bins=50, density=True, color='#4361ee', alpha=0.7, edgecolor='white')
ax2.axvline(T_max_obs, color='#f72585', lw=2.5, linestyle='--', label=f'Observed max={T_max_obs:.1f}')
ax2.set_title(f'PPC: maximum\nBayesian p-value = {pB_max:.3f}\n(near 0 → model underestimates tails)')
ax2.set_xlabel('Max of replicate'); ax2.legend(fontsize=8)

plt.suptitle('Posterior Predictive Checks: Normal model fitted to log-normal data\n'
             'Multiple test statistics reveal different aspects of misfit',
             fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Bayes factors

For models $M_1$ and $M_2$, the **Bayes factor** is the ratio of marginal likelihoods:

$$B_{12} = \frac{p(\mathbf{x} \mid M_1)}{p(\mathbf{x} \mid M_2)} = \frac{\int p(\mathbf{x}\mid\theta_1, M_1)\,p(\theta_1\mid M_1)\,d\theta_1}{\int p(\mathbf{x}\mid\theta_2, M_2)\,p(\theta_2\mid M_2)\,d\theta_2}$$

**Jeffreys' scale:**

| $\log_{10} B_{12}$ | Interpretation |
|---|---|
| 0 to 0.5 | Barely worth mentioning |
| 0.5 to 1 | Substantial evidence for $M_1$ |
| 1 to 2 | Strong evidence |
| > 2 | Decisive evidence |

**Key issues:**
- Sensitive to the prior — improper priors make BF undefined
- Computationally expensive — requires marginal likelihood
- The **Savage-Dickey density ratio** simplifies BF for nested models

## 3. Information criteria: WAIC and LOO-CV

**WAIC (Widely Applicable IC, Watanabe 2010):**
$$\text{WAIC} = -2\left(\sum_{i=1}^n \log E_\theta[p(y_i \mid \theta)] - \sum_{i=1}^n \text{Var}_\theta[\log p(y_i \mid \theta)]\right)$$

The first term is the log-predictive density; the second penalizes effective complexity. Converges to LOO-CV asymptotically.

**LOO-CV (Leave-one-out cross-validation):**
$$\text{ELPD}_{\text{LOO}} = \sum_{i=1}^n \log p(y_i \mid \mathbf{y}_{-i}) = \sum_i \log \int p(y_i \mid \theta)\,p(\theta \mid \mathbf{y}_{-i})\,d\theta$$

**PSIS-LOO** (Vehtari et al. 2017): Approximates LOO-CV from a single MCMC run using Pareto-smoothed importance weights. Computationally cheap; includes diagnostics (Pareto $k$ values).

In [ ]:
# ── LOO-CV: manual implementation for Normal model ─────────────────────────
# p(y_i | y_{-i}) approximated via importance sampling

def loo_is_normal(y, S=2000):
    """
    LOO-CV for Normal(mu, sigma^2) model with flat prior via importance sampling.
    """
    n = len(y)
    # Full posterior: mu | y ~ N(y_bar, sigma^2/n),  sigma = y.std()
    sigma    = y.std(ddof=1)
    post_mu  = y.mean()
    post_sd  = sigma / np.sqrt(n)

    mu_draws = np.random.normal(post_mu, post_sd, S)
    loo_lpd  = []

    for i in range(n):
        # IS weights: p(y_{-i}|theta) / p(y|theta) = 1/p(yi|theta)
        log_w   = -stats.norm.logpdf(y[i], mu_draws, sigma)
        log_w  -= log_w.max()
        w       = np.exp(log_w)
        w      /= w.sum()
        # LOO predictive density for y_i
        lik_i   = stats.norm.pdf(y[i], mu_draws, sigma)
        loo_lpd.append(np.log(np.sum(w * lik_i)))

    return np.array(loo_lpd)

# Compare two models on normal vs skewed data
np.random.seed(42)
y_normal = np.random.normal(5, 2, 50)
y_skewed = np.random.lognormal(1.5, 0.6, 50)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (y_data, data_label) in zip(axes, [
    (y_normal, 'Normal data'),
    (y_skewed, 'Skewed (log-normal) data'),
]):
    loo_vals = loo_is_normal(y_data)
    elpd = loo_vals.sum()

    ax.bar(range(len(loo_vals)), np.sort(loo_vals), color=[
        '#f72585' if v < np.percentile(loo_vals, 10) else '#4361ee' for v in np.sort(loo_vals)
    ], alpha=0.8, width=1.0)
    ax.set_xlabel('Observation (sorted by LOO contribution)')
    ax.set_ylabel('LOO log-predictive density')
    ax.set_title(f'{data_label}\nELPD_LOO = {elpd:.1f}\nRed bars = poor fit observations')

plt.suptitle('LOO-CV: Pointwise log-predictive density\nNormal model fits normal data well, struggles with skewed data',
             fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nModel selection summary:')
print(f'  ELPD_LOO on normal data: {loo_is_normal(y_normal).sum():.2f}  (higher = better)')
print(f'  ELPD_LOO on skewed data: {loo_is_normal(y_skewed).sum():.2f}  (lower = poor fit)')

## 4. Key takeaways

| Tool | Purpose | Key metric |
|---|---|---|
| PPC | Check model fit for specific data features | Bayesian p-value near 0 or 1 = misfit |
| Bayes factor | Compare models via marginal likelihood | $\log_{10} B > 2$ = decisive |
| WAIC | Penalized log-predictive density | Lower = better (like AIC/BIC but Bayesian) |
| PSIS-LOO | Approximate LOO-CV | ELPD_LOO; Pareto $k > 0.7$ warns bad obs. |

**Best practice (Gelman):** Use PPCs first to find misfit, then LOO-CV to compare models on predictive accuracy.

**Next:** notebook 08 — hierarchical models: the most important Bayesian modeling strategy.